# Assignment 17 - Text Cleaning, Preprocessing & NLP Pipeline

Dataset: SMS Spam Collection Dataset

Kaggle Link:
https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset

Student:
Abhishek Thakare

In [51]:
import pandas as pd
import numpy as np

import re
import string

import nltk

from nltk.corpus import stopwords

from nltk.tokenize import (
    word_tokenize,
    sent_tokenize
)

from nltk.stem import (
    PorterStemmer,
    WordNetLemmatizer
)

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\abhis\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abhis\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\abhis\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\abhis\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [52]:
df = pd.read_csv(
    "spam.csv",
    encoding="latin-1"
)

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [53]:
df = df[
    ["v1","v2"]
]

df.columns = [
    "label",
    "text"
]

In [54]:
df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [55]:
df["text_length"] = (
    df["text"]
    .apply(len)
)

df[
["text","text_length"]
].head()

,text,text_length
0,"Go until jurong point, crazy.. Available only ...",111
1,Ok lar... Joking wif u oni...,29
2,Free entry in 2 a wkly comp to win FA Cup fina...,155
3,U dun say so early hor... U c already then say...,49
4,"Nah I don't think he goes to usf, he lives aro...",61


### Raw Text Issues

While inspecting the first few messages I noticed:

- Uppercase and lowercase words
- Punctuation marks
- Numbers
- URLs
- Extra spaces
- Special characters

These issues need preprocessing before NLP analysis.

In [56]:
df["clean_text_basic"] = (
    df["text"]
    .str.lower()
)

In [57]:
df["clean_text_basic"] = (
    df["clean_text_basic"]
    .str.translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )
)

In [58]:
df["clean_text_basic"] = (
    df["clean_text_basic"]
    .str.translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )
)

In [59]:
df["clean_text_basic"] = (
    df["clean_text_basic"]
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
)

In [60]:
df[
[
"text",
"clean_text_basic"
]
].head()

,text,clean_text_basic
0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...


In [61]:
def advanced_clean(text):

    text = re.sub(
        r"http\S+",
        "",
        text
    )

    text = re.sub(
        r"<.*?>",
        "",
        text
    )

    text = re.sub(
        r"[^\w\s]",
        "",
        text
    )

    return text

In [62]:
df["clean_text_advanced"] = (
    df["clean_text_basic"]
    .apply(advanced_clean)
)

In [63]:
stop_words = set(
    stopwords.words("english")
)

In [64]:
def remove_stopwords(text):

    words = text.split()

    filtered_words = [
        word
        for word in words
        if word not in stop_words
    ]

    return " ".join(filtered_words)

In [65]:
df["text_no_stopwords"] = (
    df["clean_text_advanced"]
    .apply(remove_stopwords)
)

In [66]:
def normalize_text(text):

    text = re.sub(
        r"(.)\1+",
        r"\1",
        text
    )

    slang = {
        "u":"you",
        "gr8":"great",
        "msg":"message"
    }

    words = []

    for word in text.split():

        words.append(
            slang.get(
                word,
                word
            )
        )

    return " ".join(words)

In [67]:
df["normalized_text"] = (
    df["text_no_stopwords"]
    .apply(normalize_text)
)

In [68]:
sample_text = (
    df["normalized_text"]
    .iloc[0]
)

word_tokenize(
    sample_text
)

['go',
 'jurong',
 'point',
 'crazy',
 'available',
 'bugis',
 'n',
 'great',
 'world',
 'la',
 'e',
 'bufet',
 'cine',
 'got',
 'amore',
 'wat']

In [69]:
sent_tokenize(
    df["text"].iloc[0]
)

['Go until jurong point, crazy..',
 'Available only in bugis n great world la e buffet... Cine there got amore wat...']

In [70]:
for i in range(3):

    print(
        word_tokenize(
            df["normalized_text"]
            .iloc[i]
        )
    )

    print()

['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'bufet', 'cine', 'got', 'amore', 'wat']

['ok', 'lar', 'joking', 'wif', 'you', 'oni']

['fre', 'entry', '2', 'wkly', 'comp', 'win', 'fa', 'cup', 'final', 'tkts', '21st', 'may', '205', 'text', 'fa', '87121', 'receive', 'entry', 'questionstd', 'txt', 'ratetcs', 'aply', '0845281075over18s']



In [71]:
stemmer = PorterStemmer()

In [72]:
words = [
    "running",
    "runs",
    "runner",
    "studying"
]

for word in words:

    print(
        word,
        "->",
        stemmer.stem(word)
    )

running -> run
runs -> run
runner -> runner
studying -> studi


In [73]:
lemmatizer = (
    WordNetLemmatizer()
)

In [74]:
for word in words:

    print(
        word,
        "->",
        lemmatizer.lemmatize(word)
    )

running -> running
runs -> run
runner -> runner
studying -> studying


In [75]:
comparison = pd.DataFrame({

    "Original": words,

    "Stemmed":[
        stemmer.stem(w)
        for w in words
    ],

    "Lemmatized":[
        lemmatizer.lemmatize(w)
        for w in words
    ]
})

comparison

,Original,Stemmed,Lemmatized
0,running,run,running
1,runs,run,run
2,runner,runner,runner
3,studying,studi,studying


In [76]:
def nlp_preprocess(text):

    text = text.lower()

    text = re.sub(
        r"http\S+",
        "",
        text
    )

    text = re.sub(
        r"[^\w\s]",
        "",
        text
    )

    words = word_tokenize(text)

    words = [
        word
        for word in words
        if word not in stop_words
    ]

    words = [
        lemmatizer.lemmatize(word)
        for word in words
    ]

    return " ".join(words)

In [77]:
df["final_clean_text"] = (
    df["text"]
    .apply(nlp_preprocess)
)

In [78]:
df[
[
"text",
"final_clean_text"
]
].head()

,text,final_clean_text
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry 2 wkly comp win fa cup final tkts 2...
3,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,"Nah I don't think he goes to usf, he lives aro...",nah dont think go usf life around though


# Observations & Insights

1. While examining the raw SMS messages, I noticed inconsistent capitalization, punctuation, numbers, and extra spaces. These variations increased the complexity of the text and required preprocessing.

2. After applying basic cleaning, the text became more standardized. Converting everything to lowercase and removing punctuation reduced unnecessary variation between similar words.

3. Advanced cleaning removed URLs, special characters, and other noisy elements that are unlikely to contribute meaningful information for NLP tasks.

4. Stopword removal significantly reduced the number of words in many messages. Common words such as "the", "is", and "and" were removed, allowing the remaining words to carry more meaningful information.

5. During stemming, some words were reduced to shortened forms that were not actual dictionary words. This showed that stemming is computationally efficient but can sometimes reduce readability.

6. Lemmatization produced cleaner and more meaningful outputs because words were converted to their proper root forms while preserving dictionary validity.

7. Building a reusable NLP preprocessing pipeline helped ensure that every text sample underwent the same sequence of cleaning, tokenization, stopword removal, and lemmatization steps. This improves consistency and prepares text data for future NLP applications such as spam detection, sentiment analysis, and text classification.
